# Systematic Review
**References:**
* ❌ too old - https://github.com/chandraveshchaudhari/systematic-reviewpy
* ❌ agentic AI - https://github.com/PouriaRouzrokh/LatteReview
* ✅ Uses PubMed API - https://github.com/gijswobben/pymed

In [1]:
# Import packages
from pymed import PubMed
from dotenv import load_dotenv
import os
import json
from pathlib import Path
from datetime import datetime

# Load environment variables
load_dotenv()

True

In [ ]:
# Create a PubMed object that GraphQL can use to query
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyTool", email=os.getenv("PUBMED_EMAIL"))

In [ ]:
# Create a GraphQL query in plain text
# Hack: I used the query builder on https://pubmed.ncbi.nlm.nih.gov/advanced/ to create this

# This is the bottom-up query that guided the keywords I wanted.
#query = '"(geospatial analysis) AND (parkinson* disease) NOT (motor)"' # results = 7

# These are big-to-small queries.
#query = "((geospatial analysis) OR (geographic analysis)) AND (parkinson* disease) AND (environmental atmospheric)"
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine))" # results = 149
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine) OR (smoking))" # results = 129

# This is the final query.
query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine) OR (smoking) OR (halitosis) OR (disease-like) OR (treatment[title]) OR (neuroprotective[title]) OR (maternal) OR (preventative) OR (therapy))" # results = 86

In [ ]:
# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=100))

In [ ]:
# Loop over the retrieved articles
for article in results:

    # Print the type of object we've found (can be either PubMedBookArticle or PubMedArticle)
    print(type(article))

    # Print a JSON representation of the object
    print(article.toJSON())

In [ ]:
# Export results to NDJSON (one JSON object per line). This cell will re-run the query if `results` is missing or empty.

# Config
MAX_RESULTS = 100
OUT_DIR = Path("pubmed_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Ensure `results` is available and not an exhausted iterator
try:
    has_items = hasattr(results, "__len__") and len(results) > 0
except NameError:
    has_items = False

if not has_items:
    print("`results` is empty or undefined — running the query to fetch items")
    # Re-run the query and store as list so it can be reused
    results = list(pubmed.query(query, max_results=MAX_RESULTS))

# Timestamped filename to avoid accidental overwrites
timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
OUT = OUT_DIR / f"results-{timestamp}.ndjson"
QUERY_FILE = OUT_DIR / f"query-{timestamp}.txt"

def article_to_dict(article):
    # Prefer the object's own JSON if available
    try:
        raw = article.toJSON()
        if isinstance(raw, str):
            return json.loads(raw)
        if isinstance(raw, dict):
            return raw
    except Exception:
        pass

    # Fallback: extract common fields safely
    return {
        "pubmed_id": getattr(article, "pubmed_id", None),
        "title": getattr(article, "title", None),
        "keywords": [k for k in (getattr(article, "keywords", []) or []) if k],
        "publication_date": str(getattr(article, "publication_date", "") or ""),
        "abstract": getattr(article, "abstract", None),
    }

count = 0
with OUT.open("w", encoding="utf-8") as fh:
    for a in results:
        try:
            obj = article_to_dict(a)
            fh.write(json.dumps(obj, ensure_ascii=False))
            fh.write("\n")
            count += 1
        except Exception as e:
            # log and continue
            print(f"Failed to write article {getattr(a, 'pubmed_id', '<unknown>')}: {e}")

# Write the query metadata and query string to a timestamped text file
try:
    with QUERY_FILE.open("w", encoding="utf-8") as qf:
        qf.write(f"timestamp: {timestamp}\n")
        qf.write(f"max_results: {MAX_RESULTS}\n")
        qf.write("query:\n")
        qf.write(query)
except Exception as e:
    print(f"Failed to write query file: {e}")

print(f"Wrote {count} records to {OUT.resolve()}")
print(f"Wrote query file to {QUERY_FILE.resolve()}")


In [ ]:
# Quick verification: check results length and show up to 3 samples
try:
    if hasattr(results, '__len__'):
        print(f"results is a {type(results).__name__} with {len(results)} items")
    else:
        print(f"results is a {type(results).__name__} (no __len__)")

    # Print up to first 3 items
    for i, a in enumerate(results[:3]):
        print('\n--- Sample', i+1, '---')
        try:
            if hasattr(a, 'toJSON'):
                print(a.toJSON())
            else:
                print({
                    'pubmed_id': getattr(a, 'pubmed_id', None),
                    'title': getattr(a, 'title', None)
                })
        except Exception as e:
            print('Error printing sample:', e)
except NameError:
    print('The variable `results` is not defined. Run the query cell first.')


In [2]:
# Load the most recent NDJSON export (prefers `pubmed_results/`) and parse into a DataFrame
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path('pubmed_results')
# prefer files in OUT_DIR, fall back to cwd
candidates = list(OUT_DIR.glob('results*.ndjson')) if OUT_DIR.exists() else []
if not candidates:
    candidates = list(Path.cwd().glob('results*.ndjson'))

if not candidates:
    raise FileNotFoundError("No NDJSON files found matching 'results*.ndjson' in 'pubmed_results' or current folder. Run the export cell first.")

latest = max(candidates, key=lambda p: p.stat().st_mtime)
print(f"Loading NDJSON from: {latest.resolve()}")

records = []
with latest.open('r', encoding='utf-8') as fh:
    for i, line in enumerate(fh, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            # Show short preview of the line for debugging
            preview = line[:200] + ('...' if len(line) > 200 else '')
            print(f"Skipping line {i}: JSON decode error: {e}. Line preview: {preview!r}")

if not records:
    print("No JSON records parsed from the file.")
else:
    # Normalize into a flat DataFrame (flattens nested dicts)
    df = pd.json_normalize(records)

    # Convert keywords lists (if any) to comma-separated strings for readability
    if 'keywords' in df.columns:
        df['keywords'] = df['keywords'].apply(lambda k: ', '.join(k) if isinstance(k, (list, tuple)) else k)

    # Optional: reorder columns if common fields exist
    preferred = ['pubmed_id', 'title', 'publication_date', 'keywords', 'abstract']
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    df = df[cols]

    print(f"Loaded {len(df)} records into DataFrame with columns: {list(df.columns)}")
    display(df.head())
    # Keep df available for downstream cells
    globals()['ndjson_df'] = df
    globals()['ndjson_source_file'] = str(latest.resolve())
    print('\nYou can access the DataFrame as `ndjson_df` and the source filename as `ndjson_source_file`.')


Loading NDJSON from: /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results/results-20251028T082237Z.ndjson
Loaded 86 records into DataFrame with columns: ['pubmed_id', 'title', 'publication_date', 'keywords', 'abstract', 'authors', 'conclusions', 'copyrights', 'doi', 'journal', 'methods', 'results', 'xml']


,pubmed_id,title,publication_date,keywords,abstract,authors,conclusions,copyrights,doi,journal,methods,results,xml
0,41087353\n30879893\n36522332\n33999109\n242528...,Exposure to ambient air pollution and onset of...,2025-10-15,,This population-based longitudinal cohort stud...,"[{'affiliation': 'Queen's Business School, Que...",None,© 2025. The Author(s).,10.1038/s41531-025-01156-z,NPJ Parkinson's disease,None,None,<Element 'PubmedArticle' at 0x72c65418b790>
1,40578417,"Tris (1,3-dichloro-2-propyl) phosphate (TDCPP)...",2025-06-28,"Ferroptosis, Neuroinflammation, Oxidative stre...","Organophosphorus flame retardant TDCPP, a subs...",[{'affiliation': 'Research Center for Translat...,None,Copyright © 2025 The Authors. Published by Els...,10.1016/j.brainres.2025.149805,Brain research,None,None,<Element 'PubmedArticle' at 0x72c6541c7ce0>
2,40535452\n30287051\n38493795\n24503004\n309410...,The global rise in Parkinson's disease: a crit...,2025-06-19,"Parkinson's disease, environmental toxic pollu...",None,"[{'affiliation': 'Grupo Cerebro, Emoción y Con...",None,None,10.3389/fpubh.2025.1606732\n10.1016/S1474-4422...,Frontiers in public health,None,None,<Element 'PubmedArticle' at 0x72c6541d7830>
3,40474178\n25904081\n24976103\n38945129\n359131...,Polystyrene nanoplastics trigger pyroptosis in...,2025-06-06,"Autophagosome-lysosome fusion, Parkinson’s dis...",Parkinson's disease (PD) is a sporadic neurode...,"[{'affiliation': 'Department of Neurology, Gua...",These findings underscore how PS-NPs accelerat...,© 2025. The Author(s).,10.1186/s12967-025-06634-9,Journal of translational medicine,None,Bioluminescence imaging and Py-GCMs confirmed ...,<Element 'PubmedArticle' at 0x72c6541efec0>
4,40459174\n38493795\n39298564\n38549522\n331303...,Plastamination: A Rising Concern for Parkinson...,2025-06-03,"environment, microplastics, nanoplastics, poll...",None,"[{'affiliation': 'Department of Medicine, Surg...",None,None,10.1002/mds.30253\n10.1016/S1474-4422(24)00038...,Movement disorders : official journal of the M...,None,None,<Element 'PubmedArticle' at 0x72c6542253a0>



You can access the DataFrame as `ndjson_df` and the source filename as `ndjson_source_file`.


**TO-DO**
* Clean this notebook up
* Run the query on other databases as well
* QC the results from each database, starting with pubmed.
* Once QC'd, merge the results across each database result and process.

**REFERENCES THAT CAUGHT MY EYE**
* https://pubmed.ncbi.nlm.nih.gov/33202965/
* https://pubmed.ncbi.nlm.nih.gov/38389433/

In [ ]:
# Convert the latest NDJSON to a GitHub-friendly CSV (saved in project root)
from pathlib import Path
import pandas as pd
import json
from datetime import datetime

# Configure paths - NDJSON in pubmed_results/, CSV in project root
OUT_DIR = Path('pubmed_results')
csv_name = Path(f'results-{datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")}.csv')

# Re-use the DataFrame if it exists, otherwise load from NDJSON
if 'ndjson_df' not in globals():
    # Find latest NDJSON
    files = sorted(OUT_DIR.glob('results*.ndjson'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError("No NDJSON files found. Run the export cell first.")
    
    latest = files[0]
    print(f"Loading from {latest.name}")
    
    # Load NDJSON
    records = []
    with latest.open('r', encoding='utf-8') as fh:
        for line in fh:
            if line.strip():
                records.append(json.loads(line))
    
    # Create DataFrame
    ndjson_df = pd.json_normalize(records)
    
    # Convert keywords to strings if present
    if 'keywords' in ndjson_df.columns:
        ndjson_df['keywords'] = ndjson_df['keywords'].apply(lambda k: ', '.join(k) if isinstance(k, (list, tuple)) else k)

# Save as CSV with minimal processing for GitHub readability
try:
    # Reorder columns for readability (put common fields first)
    preferred = ['pubmed_id', 'title', 'publication_date', 'keywords', 'abstract']
    cols = [c for c in preferred if c in ndjson_df.columns] + [c for c in ndjson_df.columns if c not in preferred]
    
    # Write CSV (UTF-8 encoding, no index) to project root
    ndjson_df[cols].to_csv(csv_name, index=False, encoding='utf-8')
    print(f"Saved CSV to project root: {csv_name}")
    print(f"\nFirst few rows of the CSV:")
    display(ndjson_df[cols].head())
    
except Exception as e:
    print(f"Error saving CSV: {e}")

Saved CSV to: pubmed_results/results-20251030T015216Z.csv

First few rows of the CSV:


,pubmed_id,title,publication_date,keywords,abstract,authors,conclusions,copyrights,doi,journal,methods,results,xml
0,41087353\n30879893\n36522332\n33999109\n242528...,Exposure to ambient air pollution and onset of...,2025-10-15,,This population-based longitudinal cohort stud...,"[{'affiliation': 'Queen's Business School, Que...",None,© 2025. The Author(s).,10.1038/s41531-025-01156-z,NPJ Parkinson's disease,None,None,<Element 'PubmedArticle' at 0x72c65418b790>
1,40578417,"Tris (1,3-dichloro-2-propyl) phosphate (TDCPP)...",2025-06-28,"Ferroptosis, Neuroinflammation, Oxidative stre...","Organophosphorus flame retardant TDCPP, a subs...",[{'affiliation': 'Research Center for Translat...,None,Copyright © 2025 The Authors. Published by Els...,10.1016/j.brainres.2025.149805,Brain research,None,None,<Element 'PubmedArticle' at 0x72c6541c7ce0>
2,40535452\n30287051\n38493795\n24503004\n309410...,The global rise in Parkinson's disease: a crit...,2025-06-19,"Parkinson's disease, environmental toxic pollu...",None,"[{'affiliation': 'Grupo Cerebro, Emoción y Con...",None,None,10.3389/fpubh.2025.1606732\n10.1016/S1474-4422...,Frontiers in public health,None,None,<Element 'PubmedArticle' at 0x72c6541d7830>
3,40474178\n25904081\n24976103\n38945129\n359131...,Polystyrene nanoplastics trigger pyroptosis in...,2025-06-06,"Autophagosome-lysosome fusion, Parkinson’s dis...",Parkinson's disease (PD) is a sporadic neurode...,"[{'affiliation': 'Department of Neurology, Gua...",These findings underscore how PS-NPs accelerat...,© 2025. The Author(s).,10.1186/s12967-025-06634-9,Journal of translational medicine,None,Bioluminescence imaging and Py-GCMs confirmed ...,<Element 'PubmedArticle' at 0x72c6541efec0>
4,40459174\n38493795\n39298564\n38549522\n331303...,Plastamination: A Rising Concern for Parkinson...,2025-06-03,"environment, microplastics, nanoplastics, poll...",None,"[{'affiliation': 'Department of Medicine, Surg...",None,None,10.1002/mds.30253\n10.1016/S1474-4422(24)00038...,Movement disorders : official journal of the M...,None,None,<Element 'PubmedArticle' at 0x72c6542253a0>
